In [1]:
import networkx as nx
from rdflib import Graph, BNode, URIRef, Literal, RDF, Namespace
import rdflib
from pyvis import network as net
from tabulate import tabulate

In [ ]:
# define the table format for the output


In [2]:
# define my own schema.org namespace, since it is a bit broken in the data atm (I have fixed this, but it will only work, once the data are updated
SCHEMA = Namespace("https://schema.org")

def clean_node(node)->str:
    return str(node).split("/")[-1]
def remove_schema(part)->str:
    # handles both schema.org and schema.org/ cases
    return str(part).replace("https://schema.org", "").replace("/","")

In [3]:
g = Graph()
g.parse("uplifted_biosamples.jelly", format="jelly")

<Graph identifier=N34e9223ecc004cfdbecf4551496ab364 (<class 'rdflib.graph.Graph'>)>

In [4]:
res = g.query("SELECT ?s ?p ?o WHERE {?s ?p ?o. FILTER(!isBlank(?s))} LIMIT 5", initNs={"schema": SCHEMA})
for row in res:
    print(row)

(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Product_SAMEA112566659.jsonld'), rdflib.term.URIRef('https://schema.org/funding'), rdflib.term.URIRef('https://github.com/DerPlankton13/B5D/blob/main/GeneralSchemas/grant_b5d.jsonld'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Action_SAMEA118101326.jsonld'), rdflib.term.URIRef('https://schema.org/object'), rdflib.term.BNode('N8de7f9ad743c4f1c931d1678e82a93a7'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Product_SAMEA112550184.jsonld'), rdflib.term.URIRef('https://schema.org/additionalProperty'), rdflib.term.BNode('N44a10b30a918487e88249947d29e90b8'))
(rdflib.term.URIRef('file:///home/japlan001/PycharmProjects/metadata_converter/output/uplifted/biosamples/Product_SAMEA112585747.jsonld'), rdflib.term.URIRef('https://schema.org/keywords'), rdflib.term.BNode('N

## General Statistics

#### Number of triples in the Graph

In [5]:
print(f"{len(g):,}")

1,130,070


#### Number of Base Nodes (non-blank Nodes)

In [6]:
res = g.query(
    """
    SELECT (COUNT(DISTINCT ?s) AS ?cnt)
    WHERE {
    ?s ?p ?o .
    FILTER(!isBlank(?s))
    }
    """, initNs={"schema": SCHEMA, "rdf": RDF})
print(f"{int(next(iter(res)).cnt):,}")

13,460


#### Type Counts

In [7]:
res = g.query(
    """
    SELECT ?type (COUNT(*) AS ?cnt)
    WHERE {
    ?s rdf:type ?type .
    }
    GROUP BY ?type
    """, initNs={"schema": SCHEMA, "rdf": RDF})

rows = [(remove_schema(row.type), int(row.cnt)) for row in res]
print(tabulate(rows, headers=["Type", "Count"]))

Type               Count
---------------  -------
PropertyValue     116470
Product            16590
CreativeWork       10566
GeoCoordinates      6711
DefinedTerm        17897
HowTo               4464
ResearchProject    12705
Action              6721
Thing              13249
HowToStep           6977
Place               6721
MonetaryGrant          1


## Identify Missing values

### Empty Strings in Graph
Here I count how many strings are missing for which type

In [8]:
res = g.query(
    """
    SELECT ?p (COUNT(*) AS ?cnt)
    WHERE {?s ?p '' .}
    GROUP BY ?p ?cnt
    """,
    initNs={"schema": SCHEMA, "rdf": RDF}
)
rows = [(remove_schema(row.p), int(row.cnt)) for row in res]
print(tabulate(rows, headers=["Property", "Empty String Count"]))

Property       Empty String Count
-----------  --------------------
value                         608
description                   223


### Any Invalid Marker
Currently I look for empty strings and Literals containing the word "unknown" in both caps.

In [9]:
res = g.query(
    """
    SELECT ?type ?p ?o (COUNT(*) AS ?cnt)
    WHERE {
        ?s ?p ?o .
        ?s rdf:type ?type .
        FILTER(
            ?o = "" || REGEX(STR(?o), "unknown", "i")
        )
    }
    GROUP BY ?type ?p ?o ?cnt
    """,
    initNs={"schema": SCHEMA, "rdf": RDF}
)
rows = [(remove_schema(row.type), remove_schema(row.p), f'""' if row.o==rdflib.term.Literal('') else row.o, row.cnt) for row in res]
print(tabulate(rows, headers=["Node Type", "Property", "Object", "Count"]))


Node Type       Property     Object                                                                                                                                   Count
--------------  -----------  -------------------------------------------------------------------------------------------------------------------------------------  -------
PropertyValue   value        ""                                                                                                                                         608
Product         description  ""                                                                                                                                         223
GeoCoordinates  latitude     missing: control sample Unit unknown                                                                                                       228
GeoCoordinates  longitude    missing: control sample Unit unknown                                                                           